## Running the Experiment

This experiment is implemented as a single runnable notebook block, allowing the entire training and validation pipeline to be executed with minimal manual intervention.

When the code is executed, the following steps are performed:

1. **Library installation and import.**  
   The required libraries are installed at the beginning of the block, including transformers, datasets, accelerate, and scikit-learn. After installation, all necessary modules are imported.

2. **Device selection and seed initialization.**  
   The code automatically selects the computation device. If a CUDA-compatible GPU is available, execution is performed on GPU; otherwise, it falls back to CPU. Random seeds are then fixed to improve reproducibility across runs.

3. **Tokenizer and pretrained model setup.**  
   The tokenizer associated with bert-base-uncased is loaded, and the model is initialized. The architecture consists of a pretrained BERT encoder followed by a binary classification head. For classification, the representation of the first token ([CLS]) from the last hidden state is used.

4. **Dataset loading.**  
   The SemEval-2026 Task 13 dataset (configuration A) is downloaded and loaded automatically. 

5. **Tokenization and preprocessing.**  
   Each code sample is tokenized using the BERT tokenizer, with truncation and padding applied to a maximum sequence length of 256 tokens.

6. **Formatting and batching.**  
   The processed datasets are converted to PyTorch format and wrapped into DataLoader objects. The training set is shuffled, while the validation set is iterated in a fixed order.

7. **Freezing the encoder.**  
   In the current configuration, all BERT encoder parameters are frozen, meaning that the pretrained encoder is used as a fixed feature extractor. During training, only the classification head is updated. Although the code includes support for gradual layer unfreezing, this is not active in the current setting.

8. **Optimizer, scheduler, and loss setup.**  
   The optimizer (AdamW), a linear learning-rate scheduler with warmup, and the binary cross-entropy loss function are initialized before training begins. Separate learning rates are defined for the encoder and classification head.

9. **Checkpoint loading.**  
   If a previously saved checkpoint is found, the script restores the model, optimizer, scheduler states, and the best validation score, then resumes training from the next epoch. Otherwise, training starts from the first epoch.

10. **Training loop execution.**  
    For each epoch, the script processes the training batches, computes predictions, evaluates the loss, performs backpropagation, applies gradient clipping, updates the trainable model parameters, and steps the scheduler.

11. **Validation and metric computation.**  
    After each epoch, the model is evaluated on the validation set. Predictions are converted into binary labels using a threshold of 0.5, and macro F1 is computed as the main evaluation metric.

12. **Model and checkpoint saving.**  
    If the validation macro F1 improves, the best model weights are saved. In all cases, a checkpoint is stored after each epoch so that training can be resumed if interrupted.

## Runtime considerations

Running on CPU may significantly increase execution time. If memory limitations occur, the batch size can be reduced (for example, from 32 to 16 or 8). In addition, evaluation on the full validation set may take noticeable time depending on the available hardware.

This setup makes the experiment self-contained, facilitates reproducibility, and allows recovery in case of interruption.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

import torch
import torch.nn as nn
import random
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

dispozitiv = torch.device("cuda" if torch.cuda.is_available() else "cpu")
samanta = 42
random.seed(samanta)
np.random.seed(samanta)
torch.manual_seed(samanta)
if dispozitiv.type == "cuda":
    torch.cuda.manual_seed_all(samanta)
nume_model = "bert-base-uncased"
tokenizator = AutoTokenizer.from_pretrained(nume_model)

class BertCuDenseSimplu(nn.Module):
    def __init__(self, nume_model):
        super().__init__()
        self.bert = AutoModel.from_pretrained(nume_model)
        hidden_size = self.bert.config.hidden_size


        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 1),
            nn.ReLU(),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        iesiri = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = iesiri.last_hidden_state[:, 0]
        scoruri = self.classifier(pooled).squeeze(-1)
        return scoruri

model = BertCuDenseSimplu(nume_model)
model.to(dispozitiv)
dataset_initial = load_dataset("DaniilOr/SemEval-2026-Task13", "A")
date_antrenare = dataset_initial["train"]
date_validare = dataset_initial["validation"]
lungime_max = 256
def tokenizeaza(exemple):
    texte = exemple["code"]
    lbl = exemple["label"]
    t = tokenizator(
        texte,
        truncation=True,
        padding="max_length",
        max_length=lungime_max
    )
    t["labels"] = lbl
    return t

train_proc = date_antrenare.map(tokenizeaza, batched=True, remove_columns=["code", "generator", "language"])
valid_proc = date_validare.map(tokenizeaza, batched=True, remove_columns=["code", "generator", "language"])
train_proc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_proc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

marime_batch = 32
loader_train = DataLoader(train_proc, batch_size=marime_batch, shuffle=True)
loader_valid = DataLoader(valid_proc, batch_size=marime_batch, shuffle=False)
for p in model.bert.parameters():
    p.requires_grad = False
straturi = list(model.bert.encoder.layer)
total = len(straturi)
pas_deblocare = 0

def deblocheaza_straturi(n):
    n = max(0, min(total, n))
    start = total - n
    for i in range(total):
        strat = straturi[i]
        permite = (i >= start)
        for p in strat.parameters():
            p.requires_grad = permite

deblocheaza_straturi(pas_deblocare)
lista_enc = []
lista_clf = []
for nume, p in model.named_parameters():
    if "classifier" in nume:
        lista_clf.append(p)
    else:
        lista_enc.append(p)

lr_enc = 2e-5
lr_clf = 1e-4
nr_epoci = 3

opt = AdamW(
    [
        {"params": [x for x in lista_enc if x.requires_grad], "lr": lr_enc},
        {"params": [x for x in lista_clf if x.requires_grad], "lr": lr_clf},
    ]
)

pasi_epoca = len(loader_train)
total_pasi = pasi_epoca * nr_epoci
warm = int(total_pasi * 0.06)
scheduler = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=warm,
    num_training_steps=total_pasi
)

criteriu = nn.BCELoss()

def testeaza():
    model.eval()
    toate_pred = []
    toate_adev = []
    with torch.no_grad():
        for lot in loader_valid:
            intrari = lot["input_ids"].to(dispozitiv)
            masca = lot["attention_mask"].to(dispozitiv)
            etic = lot["labels"].to(dispozitiv)

            scoruri = model(input_ids=intrari, attention_mask=masca)

            pred_lot = (scoruri >= 0.5).long()

            toate_pred += pred_lot.cpu().numpy().tolist()
            toate_adev += etic.cpu().numpy().tolist()

    f1_macro = f1_score(toate_adev, toate_pred, average="macro")
    return f1_macro

import os

os.makedirs("checkpoints", exist_ok=True)

fisier_model = "checkpoints/bert_dense_simplu_binar.pt"
cale_checkpoint = "checkpoints/checkpoint_bert_dense_simplu_taskA.pt"

cel_mai_bun = 0.0
start_epoca = 0

if os.path.exists(cale_checkpoint):
    checkpoint = torch.load(cale_checkpoint, map_location=dispozitiv)
    model.load_state_dict(checkpoint["model_state"])
    opt.load_state_dict(checkpoint["opt_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    cel_mai_bun = checkpoint["cel_mai_bun"]
    start_epoca = checkpoint["epoca"] + 1
    print("Am gasit checkpoint. Reiau de la epoca:", start_epoca + 1)
else:
    print("Nu am gasit checkpoint. Pornesc de la epoca 1.")


for ep in range(start_epoca, nr_epoci):
    nr_deblocate = (ep + 1) * pas_deblocare
    deblocheaza_straturi(nr_deblocate)
    model.train()
    lista_pierderi = []
    for lot in loader_train:
        id_uri = lot["input_ids"].to(dispozitiv)
        masca_at = lot["attention_mask"].to(dispozitiv)
        lab = lot["labels"].to(dispozitiv).float()

        scoruri = model(input_ids=id_uri, attention_mask=masca_at)
        loss_curent = criteriu(scoruri, lab)

        lista_pierderi.append(loss_curent.item())
        loss_curent.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        scheduler.step()
        opt.zero_grad()

    scor_f1 = testeaza()
    print("Epoca:", ep + 1,
          "pierdere_medie:", float(np.mean(lista_pierderi)),
          "F1_macro_validare:", scor_f1)

    if scor_f1 > cel_mai_bun:
        cel_mai_bun = scor_f1
        torch.save(model.state_dict(), fisier_model)
        print("Model imbunatatit. F1_macro_validare:", cel_mai_bun)

    checkpoint = {
        "epoca": ep,
        "model_state": model.state_dict(),
        "opt_state": opt.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "cel_mai_bun": cel_mai_bun,
    }
    torch.save(checkpoint, cale_checkpoint)
    print("Checkpoint salvat dupa epoca", ep + 1)




## Expected Output (Illustrative Example)

During execution, the training process prints progress information for each epoch.
An example output after the first epoch is shown below (messages are displayed in Romanian, as defined in the implementation):

Nu am gasit checkpoint. Pornesc de la epoca 1.

Epoca: 1  
pierdere_medie: ~0.55  
F1_macro_validare: ~0.34  

Model imbunatatit. F1_macro_validare: ~0.34  
Checkpoint salvat dupa epoca 1  

### Interpretation

- The **average loss (~0.55)** indicates that the model begins to learn basic patterns from the data.
- The **Macro F1 score (~0.34)** represents a preliminary baseline for this task.
- Since the BERT encoder is frozen, performance is expected to be limited.

This result serves as a reference point for later experiments involving fine-tuning.